# 01 USAspending pilot: PIDP, BRIC and EDA in coastal CA/OR/WA

**Task:** T1.1 (Coastal Harbor Funding Equity)

**Reads:** `data/processed/usaspending_pilot/{prime_awards,prime_transactions,subawards}.csv` and `process_log.json`, written by `uv run python -m harborfund.ingest.usaspending all`. Also reads `data/reference/coastal_counties.csv`.

**Writes:** nothing. The outputs are the pilot report, so they are kept when committed. They hold aggregates only, no raw records.

**Scope used here:**
- PIDP and EDA: an award counts if its recipient county or place-of-performance county is one of the 42 NOAA coastal shoreline counties.
- BRIC (97.047) prime awards: every prime in CA/OR/WA counts, since the recipient is usually the state. BRIC subawards are county-flagged from ZIP codes.
- Window FY2010–FY2026 (BRIC FY2020+). Award-level tables use the base action FY. Dollar tables by FY use transaction obligations by action FY.
- Dollars are nominal.

In [1]:
import json
import numpy as np
import pandas as pd
from harborfund import config

OUT = config.PROCESSED_DIR / "usaspending_pilot"
log = json.loads((OUT / "process_log.json").read_text())
BOOL = ["in_fy_window", "has_uei", "has_duns", "recipient_is_state_gov", "prime_recipient_is_state_gov",
        "state_level_prime", "recipient_in_coastal_county", "pop_in_coastal_county", "in_coastal_county",
        "in_states", "earmark_text_flag", "earmark_candidate_year", "earmark_flag", "eda_rlf_flag",
        "harbor_kw_flag", "in_scope", "zip_multi_county", "zip_unmatched"]

def load(name):
    df = pd.read_csv(OUT / f"{name}.csv", dtype=str, low_memory=False)
    for c in BOOL:
        if c in df:
            df[c] = df[c].map({"True": True, "False": False}).fillna(False).astype(bool)
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce").astype("Int64")
    if "fema_round_fy" in df:
        df["fema_round_fy"] = pd.to_numeric(df["fema_round_fy"], errors="coerce").astype("Int64")
    for c in ("fema_program_code", "fon_family", "program_basis"):
        if c in df:
            df[c] = df[c].fillna("")
    for c in [c for c in df if c.endswith("_nominal")]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("recipient_state", "pop_state", "geo_match_basis"):
        df[c] = df[c].fillna("")
    # State the row is counted in: the matched coastal county's state; otherwise
    # (state-level BRIC primes) the recipient state if in CA/OR/WA, else the POP state.
    rec_ok = df["recipient_state"].isin(["CA", "OR", "WA"])
    df["geo_state"] = np.select(
        [df["recipient_in_coastal_county"], df["pop_in_coastal_county"], rec_ok],
        [df["recipient_state"], df["pop_state"], df["recipient_state"]], default=df["pop_state"])
    return df

awards, txns, subs = (load(n) for n in ("prime_awards", "prime_transactions", "subawards"))
print("raw run:", log["raw_run"], "| processed:", log["processed_utc"])
pd.DataFrame({k: v["raw_rows"] | {"kept_" + kk: vv for kk, vv in v["in_scope_rows"].items()}
              for k, v in log["downloads"].items()}).T

raw run: data/raw/usaspending/2026-11-18 | processed: 2026-11-18T15:02:00+00:00


,prime_awards,prime_transactions,subawards,kept_prime_awards,kept_prime_transactions,kept_subawards
aln_20.823,146,226,7,33,47,4
aln_11.300,1145,1156,36,33,33,0
aln_11.307,4284,5171,1081,193,234,32
aln_97.047,1275,2742,1360,102,236,73


## 1. Scope and award types

In [2]:
A = awards[awards.in_scope & awards.in_fy_window].copy()
T = txns[txns.in_scope & txns.in_fy_window].copy()
S = subs[subs.in_fy_window].copy()
print(f"prime awards in scope: {len(A):,} | transactions: {len(T):,} | subawards in CA/OR/WA: {len(S):,} "
      f"(in coastal counties: {S.in_coastal_county.sum():,})")
pd.crosstab(A["program_id"], A["assistance_type_description"].fillna("(blank)"), margins=True)

prime awards in scope: 334 | transactions: 550 | subawards in CA/OR/WA: 228 (in coastal counties: 109)


assistance_type_description,(blank),FORMULA GRANT (A),PROJECT GRANT (B),All
program_id,,,,
bric,0,0,47,47
eda_eaa,50,0,142,192
eda_public_works,21,0,11,32
other_20823,0,0,2,2
other_97047,0,0,2,2
pdm_competitive,0,0,19,19
pdm_cpf,0,8,1,9
pidp,0,0,31,31
All,71,8,255,334


## 2. Row counts and nominal totals by program × state × FY
Awards are counted by base action FY, and the dollars are each award's total obligation to date. The second table sums transaction obligations by action FY.

In [3]:
t_awards = (A.groupby(["program_id", "geo_state", "fy"])
              .agg(awards=("assistance_award_unique_key", "nunique"), total_obligated_nominal=("total_obligated_nominal", "sum"))
              .reset_index())
pd.set_option("display.max_rows", 400)
t_awards.pivot_table(index=["program_id", "geo_state"], columns="fy", values="awards", fill_value=0, aggfunc="sum")

fy                          2010  2011  2012  2013  2014  2015  2016  2017  \
program_id       geo_state                                                   
bric             CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
eda_eaa          CA            5     8     6     2     4     2     3     3   
                 OR            1     2     1     0     0     0     0     0   
                 WA            2     5     0     1     1     1     1     1   
eda_public_works CA            2     0     3     0     0     3     3     0   
                 OR            1     0     0     0     0     0     0     2   
                 WA            3     1     1     0     1     1     0     0   
other_20823      CA            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
other_97047      CA            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
pdm_competitive  CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
pdm_cpf          CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
pidp             CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   

fy                          2018  2019  2020  2021  2022  2023  2024  2025  \
program_id       geo_state                                                   
bric             CA            0     0     0     0     1     6     7     0   
                 OR            0     0     0     0     1     0     2     0   
                 WA            0     0     0     0    11    14     2     1   
eda_eaa          CA            5    10    23    24    16     1    10     2   
                 OR            0     0     5     2     7     0     0     0   
                 WA            0     0     9     8    13     0     3     4   
eda_public_works CA            2     0     0     2     1     1     0     1   
                 OR            0     0     1     1     1     0     0     0   
                 WA            0     0     0     0     1     0     0     0   
other_20823      CA            0     0     0     0     0     0     0     1   
                 WA            0     0     1     0     0     0     0     0   
other_97047      CA            0     0     0     1     0     0     0     0   
                 WA            0     0     0     0     1     0     0     0   
pdm_competitive  CA            0     0     4     2     1     0     1     0   
                 OR            0     0     1     1     0     0     0     0   
                 WA            0     0     3     4     2     0     0     0   
pdm_cpf          CA            0     0     0     0     1     1     0     0   
                 OR            0     0     0     0     1     0     0     0   
                 WA            0     0     0     0     3     2     1     0   
pidp             CA            0     0     0     0     2     2     6     1   
                 OR            0     0     0     0     0     1     1     1   
                 WA            0     0     0     0     1     3     2     3   

fy                          2026  
program_id       geo_state        
bric             CA            0  
                 OR            0  
                 WA            2  
eda_eaa          CA            0  
                 OR            0  
         

In [4]:
(T.groupby(["program_id", "geo_state", "fy"])["federal_action_obligation_nominal"].sum()
   .unstack("fy").fillna(0).div(1e6).round(2))  # $M nominal, by action FY

fy                          2010  2011  2012  2013  2014  2015  2016  2017  \
program_id       geo_state                                                   
bric             CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
eda_eaa          CA         4.61  8.06  5.25   1.0  2.63  2.65  1.73  1.97   
                 OR         2.75  1.32  1.76   0.0  0.50  0.00  0.00  0.00   
                 WA         0.21  2.18  0.00   2.0  0.15  0.20  0.04  0.20   
eda_public_works CA         6.30  0.00  6.75   0.0  0.00  7.10  7.74  0.00   
                 OR         0.76  0.00  0.00   0.0  0.00  0.00  0.00  4.92   
                 WA         6.20  2.60  1.00   0.0  1.50  2.00  0.00  0.00   
other_20823      CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
other_97047      CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pdm_competitive  CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pdm_cpf          CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pdm_other        CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pidp             CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   

fy                          2018   2019   2020   2021   2022   2023    2024  \
program_id       geo_state                                                    
bric             CA         0.00   0.00   0.00   0.00   0.08  11.24   35.38   
                 OR         0.00   0.00   0.00   0.00   0.13   0.00    5.11   
                 WA         0.00   0.00   0.00   0.00   1.28  24.50   22.14   
eda_eaa          CA         4.07  24.79  65.43  32.53  35.44  -4.51    6.63   
                 OR         0.00   0.00   3.50   2.24  28.87   0.00    0.00   
                 WA         0.00   0.00  22.71   9.59  87.93   0.00    3.90   
eda_public_works CA         6.00   0.00   0.00   3.86   2.66   6.02    0.00   
                 OR         0.00   0.00   1.20   2.75   0.56   0.00    0.00   
                 WA         0.00   0.00   0.00   0.00   2.25   0.00    0.00   
other_20823      CA         0.00   0.00   0.00   0.00   0.00   0.00    0.00   
                 WA         0.00   0.00   0.00   0.00   0.00  18.79    0.00   
other_97047      CA         0.00   0.00   0.00   0.19   0.00   0.00    0.00   
                 WA         0.00   0.00   0.00   0.00  -0.00   0.00    0.00   
pdm_competitive  CA         0.00   0.00   4.15   3.95  -0.83  -0.02    0.13   
                 OR         0.00   0.00   0.43   4.25  -0.15  -0.10   -0.03   
                 WA         0.00   0.00  15.09   3.43   3.94  -0.17   -0.00   
pdm_cpf          CA         0.00   0.00   0.00   0.00   2.25  12.71    0.00   
                 OR         0.00   0.00   0.00   0.00   4.43   0.00    0.00   
                 WA         0.00   0.00   0.00   0.00  13.00  10.90    5.66   
pdm_other        CA         0.00   0.00  -0.09   0.00   0.00  -0.14    0.00   
pidp             CA         0.00   0.00   0.00   0.00  32.68   9.88  106.57   
                 OR         0.00   0.00   0.00   0.00   0.00   9.88    3.44   
                 WA         0.00   0.00   0.00   0.00  10.69  22.58   27.94   

fy                           2025    2026  
program_id       geo_state   

In [5]:
summary = (A.groupby(["program_id", "geo_state"])
             .agg(awards=("assistance_award_unique_key", "nunique"),
                  obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6))
             .round(2))
summary

awards  obligated_m
program_id       geo_state                     
bric             CA             14        50.39
                 OR              3         5.80
                 WA             30        98.52
eda_eaa          CA            124       194.75
                 OR             18        40.21
                 WA             50       144.07
eda_public_works CA             18        50.39
                 OR              6        10.18
                 WA              8        15.55
other_20823      CA              1         5.55
                 WA              1         1.50
other_97047      CA              1         0.19
                 WA              1        -0.00
pdm_competitive  CA              8         2.25
                 OR              2         8.79
                 WA              9        19.37
pdm_cpf          CA              2        14.96
                 OR              1         4.43
                 WA              6        39.50
pidp             CA             16       265.85
                 OR              3        21.04
                 WA             12       170.42

## 3. Recipient identifiers (UEI / legacy DUNS)
SAM.gov switched from DUNS to UEI in April 2022, so older awards may have only a DUNS.

In [6]:
ids = A.assign(uei_or_duns=A.has_uei | A.has_duns)
pd.concat([
    ids.groupby("program_id")[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3),
    ids.groupby("program_id").size().rename("awards"),
], axis=1)

,has_uei,has_duns,uei_or_duns,awards
program_id,,,,
bric,1.0,0.064,1.0,47
eda_eaa,1.0,0.646,1.0,192
eda_public_works,1.0,0.875,1.0,32
other_20823,1.0,0.000,1.0,2
other_97047,1.0,1.000,1.0,2
pdm_competitive,1.0,0.421,1.0,19
pdm_cpf,1.0,0.000,1.0,9
pidp,1.0,0.032,1.0,31


In [7]:
ids.assign(era=pd.cut(ids.fy.astype(float), [2009, 2021, 2026], labels=["FY2010-21", "FY2022-26"])) \
   .groupby(["program_id", "era"], observed=True)[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3)

has_uei  has_duns  uei_or_duns
program_id       era                                      
bric             FY2022-26      1.0     0.064          1.0
eda_eaa          FY2010-21      1.0     0.881          1.0
                 FY2022-26      1.0     0.088          1.0
eda_public_works FY2010-21      1.0     1.000          1.0
                 FY2022-26      1.0     0.200          1.0
other_20823      FY2010-21      1.0     0.000          1.0
                 FY2022-26      1.0     0.000          1.0
other_97047      FY2010-21      1.0     1.000          1.0
                 FY2022-26      1.0     1.000          1.0
pdm_competitive  FY2010-21      1.0     0.467          1.0
                 FY2022-26      1.0     0.250          1.0
pdm_cpf          FY2022-26      1.0     0.000          1.0
pidp             FY2022-26      1.0     0.032          1.0

In [8]:
sub_ids = S[S.in_coastal_county]
sub_ids.groupby("program_id")[["has_uei", "has_duns"]].mean().round(3).assign(subawards=sub_ids.groupby("program_id").size())

,has_uei,has_duns,subawards
program_id,,,
bric,1.0,1.000,38
eda_eaa,1.0,0.781,32
pdm_competitive,1.0,1.000,20
pdm_cpf,1.0,0.933,15
pidp,1.0,1.000,4


## 4. BRIC: state-held prime dollars and what subawards identify

In [9]:
B = A[A.aln == "97.047"]
bric_state = B.groupby("recipient_is_state_gov")["total_obligated_nominal"].agg(["count", "sum"])
bric_state["share_of_dollars"] = (bric_state["sum"] / bric_state["sum"].sum()).round(3)
bric_state

,count,sum,share_of_dollars
recipient_is_state_gov,,,
False,45,2.532256e+07,0.104
True,32,2.188741e+08,0.896


In [10]:
state_primes = B[B.recipient_is_state_gov]
sb = subs[(subs.aln == "97.047") & subs.prime_award_unique_key.isin(state_primes.assistance_award_unique_key)]
rows = []
for st, g in state_primes.groupby("recipient_state"):
    s_all = sb[sb.prime_award_unique_key.isin(g.assistance_award_unique_key)]
    s_coast = s_all[s_all.in_coastal_county]
    prime_d = g.total_obligated_nominal.sum()
    rows.append({"state": st, "state_primes": len(g), "prime_obligated_m": prime_d / 1e6,
                 "subawards": len(s_all), "subaward_m": s_all.subaward_amount_nominal.sum() / 1e6,
                 "share_identified_by_subawards": s_all.subaward_amount_nominal.sum() / prime_d if prime_d else None,
                 "coastal_subawards": len(s_coast), "coastal_subaward_m": s_coast.subaward_amount_nominal.sum() / 1e6})
pd.DataFrame(rows).round(3)

,state,state_primes,prime_obligated_m,subawards,subaward_m,share_identified_by_subawards,coastal_subawards,coastal_subaward_m
0,CA,11,63.874,50,57.610,0.902,33,41.902
1,OR,6,19.017,16,11.037,0.580,4,4.334
2,WA,15,135.983,42,118.457,0.871,24,100.542


Caveat: subaward amounts are FFATA-reported and can be double-reported or amended, so the identified share may go above 1. The prime total is the obligation to date.

In [11]:
sub_geo = S[S.aln == "97.047"]
pd.Series({"subawards_in_states": len(sub_geo), "zip_unmatched": sub_geo.zip_unmatched.sum(),
           "zip_multi_county": sub_geo.zip_multi_county.sum(), "in_coastal_county": sub_geo.in_coastal_county.sum()})

subawards_in_states    133
zip_unmatched           10
zip_multi_county        13
in_coastal_county       73
dtype: int64

## 5. Program split and earmark flags
- **97.047** is classified from the FEMA award ID, e.g. `EMS-2022-PD-0002` → round FY2022, code `PD`. `BR` is BRIC, `PC` is legacy PDM competitive, and `PD` from FY2022 is the PDM congressionally directed (earmark) award. Descriptions aren't used because they're unreliable: PDM earmark awards are often described as "BRIC", and one is labelled "Emergency Management Performance Grants".
- **20.823** is classified from the funding-opportunity family. `MA-PID-yy` is a PIDP round. `MARAD BAA 2019/2020` covers awards from 2019/2020 broad agency announcements, probably the early PIDP rounds (that mapping is unverified). Marine Highway (AMHP) and DOT BUILD/INFRA records are split off as `other_20823`.
- No 20.823 record says Community Project Funding, so PIDP earmarks (FY2024, FY2026) **can't** be separated with USAspending alone. T2.4's THUD tables are needed for that.

In [12]:
E = A[A.aln.isin(["20.823", "97.047"])]
(E.groupby(["aln", "program_id", "program_basis"])
   .agg(awards=("assistance_award_unique_key", "nunique"), obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6),
        earmark_flagged=("earmark_flag", "sum"))
   .round(2))

awards  obligated_m  \
aln    program_id      program_basis                             
20.823 other_20823     text:AMHP                2         7.05   
       pidp            fon:MA-PID-21            3        70.47   
                       fon:MA-PID-22           10       149.12   
                       fon:MA-PID-23            6        87.33   
                       fon:MA-PID-24            5        80.41   
                       fon:MARAD BAA 2019       2        32.68   
                       fon:MARAD BAA 2020       2        16.72   
                       fon:blank                3        20.57   
97.047 bric            fain:BR                 47       154.71   
       other_97047     fain:CA                  1         0.19   
                       fain:GR                  1        -0.00   
       pdm_competitive fain:PC                 19        30.41   
       pdm_cpf         fain:PD                  9        58.89   

                                           earmark_flagged  
aln    program_id      program_basis                        
20.823 other_20823     text:AMHP                         0  
       pidp            fon:MA-PID-21                     0  
                       fon:MA-PID-22                     0  
                       fon:MA-PID-23                     0  
                       fon:MA-PID-24                     0  
                       fon:MARAD BAA 2019                0  
                       fon:MARAD BAA 2020                0  
                       fon:blank                         0  
97.047 bric            fain:BR                           0  
       other_97047     fain:CA                           0  
                       fain:GR                           0  
       pdm_competitive fain:PC                           0  
       pdm_cpf         fain:PD                           9

In [13]:
F = awards[awards.aln == "97.047"]
pd.crosstab([F.fema_program_code.replace("", "?"), F.in_scope], F.fema_round_fy.astype("string").fillna("?"))

fema_round_fy               2008  2010  2011  2012  2013  2014  2015  2016  \
fema_program_code in_scope                                                   
BR                False        0     0     0     0     0     0     0     0   
                  True         0     0     0     0     0     0     0     0   
CA                True         0     0     0     0     0     0     0     0   
GR                True         0     0     0     0     1     0     0     0   
PC                False        0     1     1     0     0     1     1     2   
                  True         0     0     1     0     1     2     0     4   
PD                False        0     0     0     1     0     0     0     0   
                  True         1     1     0     0     0     0     0     0   

fema_round_fy               2017  2018  2019  2020  2021  2022  2023  2024  
fema_program_code in_scope                                                  
BR                False        0     0     0     1     2     4     2     0  
                  True         0     0     0    28    13     5     1     0  
CA                True         0     0     0     0     1     0     0     0  
GR                True         0     0     0     0     0     0     0     0  
PC                False        2     4     1     5     1     1     0     0  
                  True         4     7     4     9     6     3     0     1  
PD                False        0     0     0     0     0     0     0     0  
                  True         0     0     0     0     0     5     3     1

In [14]:
P = awards[awards.aln == "20.823"]
pd.crosstab(P.fon_family, P.fy)

fy,2020,2022,2023,2024,2025,2026
fon_family,,,,,,
MA-PID-21,0,0,1,2,0,0
MA-PID-22,0,0,1,7,1,3
MA-PID-23,0,0,0,2,3,1
MA-PID-24,0,0,0,0,1,4
MARAD BAA 2019,0,2,0,0,0,0
MARAD BAA 2020,1,0,2,0,0,0
blank,0,1,2,0,0,0
other,0,0,0,0,2,0


In [15]:
E[E.earmark_flag][["aln", "program_id", "award_id_fain", "fy", "recipient_name", "geo_state", "in_coastal_county",
                   "total_obligated_nominal", "prime_award_base_transaction_description"]].sort_values(["aln", "fy"])

,aln,program_id,award_id_fain,fy,recipient_name,geo_state,in_coastal_county,total_obligated_nominal,prime_award_base_transaction_description
615,97.047,pdm_cpf,EMF-2022-PD-0001,2022,OFFICE OF EMERGENCY SERVICES,CA,False,2252500.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
685,97.047,pdm_cpf,EMS-2022-PD-0001,2022,"MILITARY DEPARTMENT, WASHINGTON STATE",WA,True,19900000.00,EMERGENCY MANAGEMENT PERFORMANCE GRANTS
686,97.047,pdm_cpf,EMS-2022-PD-0002,2022,OREGON DEPARTMENT OF EMERGENCY MANAGEMEN,OR,False,4426406.25,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
687,97.047,pdm_cpf,EMS-2022-PD-0003,2022,HOH INDIAN TRIBE,WA,True,1650000.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
688,97.047,pdm_cpf,EMS-2022-PD-0004,2022,QUINAULT INDIAN NATION,WA,True,1400000.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
618,97.047,pdm_cpf,EMF-2023-PD-0004,2023,OFFICE OF EMERGENCY SERVICES,CA,False,12705936.34,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
690,97.047,pdm_cpf,EMS-2023-PD-0003,2023,"MILITARY DEPARTMENT, WASHINGTON STATE",WA,True,8927058.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
691,97.047,pdm_cpf,EMS-2023-PD-0004,2023,HOH INDIAN TRIBE,WA,True,1968750.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...
692,97.047,pdm_cpf,EMS-2024-PD-0003,2024,"MILITARY DEPARTMENT, WASHINGTON STATE",WA,True,5655070.00,BRIC: BUILDING RESILIENT INFRASTRUCTURE AND CO...


In [16]:
# Subawards under PDM earmark primes carry LPDM project IDs, which corroborates the FAIN rule
sub_cpf = subs[subs.program_id == "pdm_cpf"]
pd.Series({"subawards": len(sub_cpf),
           "with LPDM project id": sub_cpf.subaward_description.fillna("").str.contains(r"^LPDM\d{2}", regex=True).sum(),
           "in coastal county": sub_cpf.in_coastal_county.sum(),
           "coastal $M": round(sub_cpf.loc[sub_cpf.in_coastal_county, "subaward_amount_nominal"].sum() / 1e6, 2)})

subawards               21.00
with LPDM project id    12.00
in coastal county       15.00
coastal $M              29.24
dtype: float64

## 6. Harbor keyword flag and geography match basis
`harbor_kw_flag` applies only the inventory's keyword rule, matched at word starts (so 'port' doesn't match 'support'). The crosswalk part of the rule comes later (T1.2/T1.3). PIDP is `all` (a port program). BRIC/PDM prime descriptions are generic program text, so the keyword rule can only work on subaward descriptions, and OpenFEMA (T2.5a) has the project titles.

In [17]:
pd.concat([A.groupby("program_id")["harbor_kw_flag"].agg(["sum", "mean"]).rename(columns={"sum": "flagged", "mean": "share"}),
           A[A.harbor_kw_flag].groupby("program_id")["total_obligated_nominal"].sum().div(1e6).rename("flagged_m")], axis=1).round(3)

,flagged,share,flagged_m
program_id,,,
bric,0,0.000,NaN
eda_eaa,16,0.083,56.067
eda_public_works,5,0.156,7.288
other_20823,0,0.000,NaN
other_97047,0,0.000,NaN
pdm_competitive,0,0.000,NaN
pdm_cpf,0,0.000,NaN
pidp,31,1.000,457.314


In [18]:
pd.crosstab(A.program_id, A.geo_match_basis.replace("", "(state-level BRIC prime)"), margins=True)

geo_match_basis,(state-level BRIC prime),both,pop,recipient,All
program_id,,,,,
bric,12,35,0,0,47
eda_eaa,0,172,5,15,192
eda_public_works,0,32,0,0,32
other_20823,0,2,0,0,2
other_97047,0,2,0,0,2
pdm_competitive,2,17,0,0,19
pdm_cpf,3,6,0,0,9
pidp,0,31,0,0,31
All,17,297,5,15,334


In [19]:
if "eda_rlf_flag" in A:
    print(A[A.aln.isin(["11.300", "11.307"])].groupby("program_id")["eda_rlf_flag"].sum())

program_id
eda_eaa             30
eda_public_works     0
Name: eda_rlf_flag, dtype: int64


## 7. Independent count checks
One program-year per program, compared with a published figure. All checks use every award in the state, not just coastal counties, because the published figures are statewide.

| Program | External figure | Source (read 2026-11-18) |
| --- | --- | --- |
| PIDP | MARAD FY2022 PIDP award list: 12 awards in CA/OR/WA | https://www.maritime.dot.gov/sites/marad.dot.gov/files/2022-10/FY%202022%20Port%20Infrastructure%20Development%20Grant%20Awards.pdf |
| EDA Public Works | EDA FY2014 annual report, Appendix 2: CA 2 / $5,620,500; OR 1 / $1,580,800; WA 4 / $5,400,000 | https://eda.gov/sites/default/files/2024-11/Appendix-2-fy2014-Summary-of-EDA-Fiscal-Investments-by-State.pdf |
| EDA EAA (reference only) | Same appendix: CA 5 / $1,985,432; OR 3 / $901,036; WA 3 / $1,721,415 | same |
| BRIC | Oregon OEM, March 2026: "Oregon's single awarded FY2020 project" | https://apps.oregon.gov/oregon-newsroom/OR/OEM/Posts/Post/BRIC-update-March-2026 |

The MARAD figures were read through a text-extraction tool, so the project names and amounts below are as extracted.

In [20]:
marad_fy22 = pd.DataFrame([
    ("CA", "Fisherman's Terminal Piling Replacement (Eureka)", 650000),
    ("CA", "Middle Harbor Terminal Zero Emission Conversion (Long Beach)", 30141080),
    ("CA", "Outer Harbor Terminal Redevelopment (Oakland)", 36592875),
    ("CA", "Amador Street Infrastructure Improvement (San Francisco)", 9607500),
    ("CA", "Port of Stockton Rail Rehabilitation & Upgrade", 9628477),
    ("CA", "Seawall Replacement (Crescent City)", 7366566),
    ("OR", "Terminal 6 Infrastructure Improvements (Portland)", 24360000),
    ("WA", "Terminal 5 Export, Expansion, and Emissions Reduction (Seattle)", 17035900),
    ("WA", "Grays Harbor Terminal 4 Expansion & Redevelopment (Aberdeen)", 25500000),
    ("WA", "Intermodal Handling and Transfer Facility Improvements (Port Angeles)", 8608000),
    ("WA", "Master Plan, Pier Design, and Boat Ramp Construction (La Conner)", 11073627),
    ("WA", "Seaport Throughput Improvement (Olympia)", 9270918),
], columns=["state", "project", "marad_amount"])
us22 = awards[(awards.fon_family == "MA-PID-22")][["recipient_name", "recipient_state", "in_coastal_county", "total_obligated_nominal"]]
chk = marad_fy22.merge(us22, left_on="marad_amount", right_on="total_obligated_nominal", how="outer", indicator=True)
print("MARAD FY2022 CA/OR/WA:", len(marad_fy22), "| USAspending MA-PID-22 in CA/OR/WA:", len(us22),
      "| exact-amount matches:", (chk._merge == "both").sum())
chk.sort_values(["_merge", "state"])

MARAD FY2022 CA/OR/WA: 12 | USAspending MA-PID-22 in CA/OR/WA: 12 | exact-amount matches: 9


,state,project,marad_amount,recipient_name,recipient_state,in_coastal_county,total_obligated_nominal,_merge
14,CA,Outer Harbor Terminal Redevelopment (Oakland),36592875.0,NaN,NaN,NaN,NaN,left_only
7,WA,"Master Plan, Pier Design, and Boat Ramp Constr...",11073627.0,NaN,NaN,NaN,NaN,left_only
8,WA,"Terminal 5 Export, Expansion, and Emissions Re...",17035900.0,NaN,NaN,NaN,NaN,left_only
1,NaN,NaN,NaN,PORT DEPARTMENT OF THE CITY OF OAKLAND,CA,True,5200000.0,right_only
9,NaN,NaN,NaN,THE NORTHWEST SEAPORT ALLIANCE,WA,True,17308440.0,right_only
13,NaN,NaN,NaN,PORT DEPARTMENT OF THE CITY OF OAKLAND,CA,True,35467875.0,right_only
0,CA,Fisherman's Terminal Piling Replacement (Eureka),650000.0,CITY OF EUREKA,CA,True,650000.0,both
2,CA,Seawall Replacement (Crescent City),7366566.0,CRESCENT CITY HARBOR DISTRICT,CA,True,7366566.0,both
5,CA,Amador Street Infrastructure Improvement (San ...,9607500.0,CITY & COUNTY OF SAN FRANCISCO,CA,True,9607500.0,both
6,CA,Port of Stockton Rail Rehabilitation & Upgrade,9628477.0,STOCKTON PORT DISTRICT,CA,False,9628477.0,both


Unmatched rows, as of the 2026-11-18 pull:
- **Swinomish (La Conner, $11,073,627)** is in the national file, but USAspending gives the recipient and the place of performance as Superior, WI. No address filter can find it, and a UEI crosswalk would. That's the crosswalk-first rule working as intended.
- **Seattle T5** shows $17,308,440 against MARAD's $17,035,900, and **Oakland** $35,467,875 against $36,592,875. These are the same projects with amended obligations. The difference is $272,540 for Seattle and $1,125,000 for Oakland.
- **Oakland $5.2M** (substation, FY2024) carries a `MA PID-22 001` number but isn't on MARAD's FY2022 list. The number may be a typo for another round. Not checked.

In [21]:
eda14 = (awards[(awards.aln.isin(["11.300", "11.307"])) & (awards.fy == 2014) & awards.recipient_state.isin(["CA", "OR", "WA"])]
         .groupby(["program_id", "recipient_state"])["total_obligated_nominal"].agg(["count", "sum"]))
ext = pd.DataFrame([("eda_public_works", "CA", 2, 5620500), ("eda_public_works", "OR", 1, 1580800),
                    ("eda_public_works", "WA", 4, 5400000), ("eda_eaa", "CA", 5, 1985432),
                    ("eda_eaa", "OR", 3, 901036), ("eda_eaa", "WA", 3, 1721415)],
                   columns=["program_id", "recipient_state", "eda_count", "eda_sum"])
eda14.reset_index().merge(ext, on=["program_id", "recipient_state"], how="outer")

,program_id,recipient_state,count,sum,eda_count,eda_sum
0,eda_eaa,CA,7,3669125.0,5,1985432
1,eda_eaa,OR,2,401036.0,3,901036
2,eda_eaa,WA,2,995000.0,3,1721415
3,eda_public_works,CA,2,5620500.0,2,5620500
4,eda_public_works,OR,1,1580800.0,1,1580800
5,eda_public_works,WA,4,5400000.0,4,5400000


In [22]:
or20 = awards[(awards.aln == "97.047") & (awards.fema_program_code == "BR") & (awards.fema_round_fy == 2020) & (awards.recipient_state == "OR")]
print("Oregon FY2020-round BRIC prime awards in USAspending:", len(or20), "| Oregon OEM: 1")
or20[["award_id_fain", "recipient_name", "total_obligated_nominal"]]

Oregon FY2020-round BRIC prime awards in USAspending: 1 | Oregon OEM: 1


,award_id_fain,recipient_name,total_obligated_nominal
652,EMS-2020-BR-131-0006,OREGON MILITARY DEPARTMENT,127500.0
